## Responses API란 무엇인가요?

Responses API는 OpenAI 모델과 상호작용하는 새로운 방식으로, 이전 API보다 더 간단하고 유연하게 설계되었습니다. 이 API는 여러 도구를 사용하고, 다중 턴 대화를 처리하며, 텍스트뿐만 아니라 다양한 유형의 데이터를 다룰 수 있는 고급 AI 애플리케이션을 쉽게 구축할 수 있도록 합니다.

Chat Completions와 같이 주로 텍스트를 위해 설계된 이전 API나 설정이 많이 필요한 Assistants API와 달리, Responses API는 다음을 위해 처음부터 설계되었습니다:

- 원활한 다중 턴 상호작용 (단일 API 호출에서 여러 단계의 대화를 이어갈 수 있음)
- 강력한 호스팅 도구에 대한 쉬운 접근 (파일 검색, 웹 검색, 코드 해석기 등)
- 모델에 보낼 컨텍스트에 대한 세밀한 제어

AI 모델이 점점 더 복잡하고 장기적인 추론을 수행할 수 있게 되면서, 비동기적이고 상태를 유지할 수 있는 API가 필요합니다. Responses API는 이러한 요구를 충족하도록 설계되었습니다.

이 가이드에서는 Responses API가 제공하는 새로운 기능과 함께 시작하는 데 도움이 되는 실용적인 예제를 살펴봅니다.

## 기본 사항

Responses API는 표면적으로 Completions API와 매우 유사하게 설계되었습니다.

In [1]:
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import os
from dotenv import load_dotenv
load_dotenv(override=True)

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

client = AzureOpenAI(
  azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"), 
  azure_ad_token_provider=token_provider,
  api_version="2025-03-01-preview"
)

In [2]:
response = client.responses.create(
    model="gpt-4.1",
    input="농담 하나 해줘",
)

In [3]:
print(response.output[0].content[0].text)

물론이죠! 여기 한국식 농담 하나 있습니다:

왜 토끼는 공부를 못할까요?

  
**답:**  
  
계속 당근(당근, "당연히"와 "당근"의 발음이 같음)을 찾느라! 😂


Responses API의 주요 기능 중 하나는 상태를 유지한다는 점입니다. 즉, 대화의 상태를 직접 관리할 필요가 없으며, API가 이를 대신 처리합니다. 예를 들어, 언제든지 응답을 가져오면 전체 대화 기록이 포함됩니다.

In [4]:
fetched_response = client.responses.retrieve(
    response_id=response.id
)

print(fetched_response.output[0].content[0].text)

물론이죠! 여기 한국식 농담 하나 있습니다:

왜 토끼는 공부를 못할까요?

  
**답:**  
  
계속 당근(당근, "당연히"와 "당근"의 발음이 같음)을 찾느라! 😂


이전 응답을 참조하여 대화를 계속할 수 있습니다.

In [5]:
response_two = client.responses.create(
    model="gpt-4.1",
    input="다른 농담 하나 더 해줘",
    previous_response_id=response.id
)

In [6]:
print(response_two.output[0].content[0].text)

알겠습니다! 이번엔 새 농담입니다:

  
**Q:** 바다가 좋아하는 옷은 뭘까요?

**A:**  
  
"해" 지는 옷! 😄

(‘해’가 바다와 관련 있고, ‘해지는 옷’은 해가 지는 옷이라서 두 가지 의미!)


물론 컨텍스트를 직접 관리할 수도 있습니다. 하지만 OpenAI가 컨텍스트를 유지해 주는 이점 중 하나는 응답을 원하는 시점에서 분기(fork)하여 해당 지점에서 대화를 계속할 수 있다는 점입니다.

In [7]:
response_two_forked = client.responses.create(
    model="gpt-4.1",
    input="그 농담은 별로였어. 다른 농담 하나 더 해주고 두 농담의 차이점도 알려줘",
    previous_response_id=response.id  # 첫 번째 응답에서 분기하여 계속 진행
)

output_text = response_two_forked.output[0].content[0].text
print(output_text)

알겠습니다! 좀 더 다른 스타일의 농담을 해볼게요:

**농담 2:**  
호랑이가 학교에 간 이유는 무엇일까요?

  
**답:**  
범(범, "범"은 호랑이를 뜻하는 말이면서 "범"은 범위를 뜻함)생(학생)이 되려고! 😂

---

### 두 농담의 차이점 설명

- **첫 번째 농담(토끼, 당근):**
  - 토끼와 당근의 관계를 소재로 삼았습니다.
  - "당근(당연히/실제로 당근)"이라는 단어의 발음을 이용한 언어유희(말장난)를 사용했어요.
  - 단순한 어휘 속성에 기반해 웃음을 주려 합니다.

- **두 번째 농담(호랑이, 범생이):**
  - 호랑이와 학생("범생이")라는 단어를 연결했습니다.
  - "범생이"라는 단어가 '범(호랑이)'과 '생이(학생)'를 결합하는 중의적 표현을 사용했어요.
  - 동물 이름과 인간의 속성을 결합해 상황 유머를 만들었습니다.

즉, 두 농담 모두 언어유희를 이용하지만,  
첫 번째는 일상적인 어휘(당근)의 발음에 초점을 두었고,  
두 번째는 단어의 뜻(범, 학생)이 합쳐져 새로운 의미를 만들어내는 방식입니다! 

혹시 더 듣고 싶으시면 말씀해주세요 :)


## 호스팅 도구

Responses API의 또 다른 이점은 `file_search` 및 `web_search`와 같은 호스팅 도구를 지원한다는 점입니다. 도구를 수동으로 호출하는 대신, 도구를 전달하면 API가 어떤 도구를 사용할지 결정하고 이를 사용합니다.

다음은 `web_search` 도구를 사용하여 웹 검색 결과를 응답에 통합하는 예제입니다.

In [14]:
import time
import json

response = client.responses.create(
    model="gpt-4.1",
    input="AI에 대한 최신 뉴스는 무엇인가요?",
    tools=[{"type": "web_search"}],  # 환경에 따라 web_search_preview 필요 가능
)

print("initial status:", response.status)

for _ in range(20):
    if response.status == "completed":
        break
    time.sleep(1.5)
    response = client.responses.retrieve(response.id)
    print("poll status:", response.status)

print("final status:", response.status)

initial status: completed
final status: completed


In [15]:
import json
if hasattr(response, "output_text") and response.output_text:
    print(response.output_text)
else:
    print(json.dumps(response.output, default=lambda o: o.__dict__, indent=2, ensure_ascii=False))

2026년 3월 기준, 최근 AI(인공지능) 분야에서 주목할 만한 뉴스와 트렌드는 다음과 같습니다.

---

**1. 차세대 초대형 생성형 AI 모델 경쟁**
- 오픈AI가 GPT-5.4를 공개했습니다. 이 모델은 기존 모델보다 더 강화된 추론과 코딩 능력, 그리고 본격적인 '에이전트 AI(스스로 행동하는 AI)' 기능을 포함해, 복잡한 업무를 통합적으로 수행할 수 있다는 점이 주요 특징입니다. 구글 역시 제미나이 3.1을 발표하며 생성형 AI 경쟁이 심화되고 있습니다. [인공지능신문](https://www.aitimes.kr/news/articleView.html?idxno=49669)

---

**2. 에이전트형(Agentic) AI와 피지컬(Physical) AI의 확산**
- 이제 AI는 단순 생성에서 벗어나 사용자 지시에 따라 계획을 세우고 실행까지 하는 단계(예: 항공권 예매, 결제 등)로 진화 중입니다. BMW 등은 휴머노이드 로봇 등 피지컬 AI를 실물 공장에 투입하며, 생산 현장에까지 적용이 확대되고 있습니다. [연합뉴스](https://www.yna.co.kr/view/AKR20251231030700017)

---

**3. AI 관련 법·정책 변화와 사회적 고민**
- 2026년 한국 ‘AI 기본법’ 등 AI의 영향력이 큰 분야(의료, 에너지, 교통 등)는 보다 엄격한 관리와 규제 적용이 시작됩니다. 에이전트 AI가 발생시킨 오류나 사고의 책임 문제도 사회적 이슈로 부상 중입니다. [연합뉴스](https://www.yna.co.kr/view/AKR20251231030700017)

---

**4. AI 인프라(반도체, 에너지) 경쟁과 기술 독립 움직임**
- AI의 연산을 담당하는 반도체·GPU 시장에서 엔비디아의 독점 구도가 다소 흔들리고 있습니다. 구글, 아마존, 메타 등이 독자 AI칩 개발에 박차를 가하며, 반도체 시장 경쟁도 치열해지고 있습니다. 또한 전 세계적으로 데이터센터 전력 수요가 폭증하면서 에너지 확보도 큰 과제가 되

## 멀티모달, 도구 확장 대화

Responses API는 텍스트, 이미지, 오디오 모달리티를 기본적으로 지원합니다.  
모든 것을 결합하여 Responses API를 통해 단일 API 호출로 완전한 멀티모달, 도구 확장 상호작용을 구축할 수 있습니다.

In [16]:
import os
import json
import base64
import mimetypes

import requests
from IPython.display import Image, display

# 제공된 URL에서 이미지를 표시
# url = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Cat_August_2010-4.jpg/2880px-Cat_August_2010-4.jpg"
url = "https://images.pexels.com/photos/104827/cat-pet-animal-domestic-104827.jpeg"
display(Image(url=url, width=400))

# 1) 내 환경(브라우저처럼)에서 다운로드 (여기서는 UA를 줄 수 있음)
headers = {"User-Agent": "azure-ai-workshop/1.0 (contact: you@example.com)"}
r = requests.get(url, headers=headers, timeout=30)
r.raise_for_status()
img_bytes = r.content

# 2) mime 추정 + base64 data URL 생성
mime, _ = mimetypes.guess_type(url)
mime = mime or "image/jpeg"
data_url = f"data:{mime};base64,{base64.b64encode(img_bytes).decode()}"

response_multimodal = client.responses.create(
    model="gpt-4.1",
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "이 이미지와 관련된 키워드 10개를 한국어로 생성해줘."},
            {"type": "input_image", "image_url": data_url}
        ]
    }],
    tools=[
        {"type": "web_search"}
    ]
)

In [19]:
import json

def print_response_text(resp):
    # 1) 가장 간단한 경로: SDK가 합쳐준 텍스트 제공
    if getattr(resp, "output_text", None):
        print(resp.output_text)
        return

    # 2) output 배열에서 message/output_text를 수집
    extracted = []
    for item in getattr(resp, "output", []) or []:
        if getattr(item, "type", None) == "message":
            for content in getattr(item, "content", []) or []:
                if getattr(content, "type", None) == "output_text":
                    text = getattr(content, "text", "")
                    if text:
                        extracted.append(text)

    if extracted:
        print("\n\n".join(extracted))
    else:
        # 3) 마지막 fallback: 한글이 보이도록 ensure_ascii=False
        print(json.dumps(resp.__dict__, default=lambda o: o.__dict__, indent=2, ensure_ascii=False))

print_response_text(response_multimodal)

1. 고양이  
2. 귀  
3. 동물  
4. 털  
5. 귀여움  
6. 반려동물  
7. 하얀 배경  
8. 초점  
9. 사랑스러움  
10. 촬영


위 예제에서는 `web_search` 도구를 사용하여 단일 API 호출로 이미지와 관련된 뉴스를 검색할 수 있었습니다. 이는 Chat Completions API를 사용할 경우 여러 번의 왕복 호출이 필요했던 작업을 단순화합니다.

Responses API를 사용하면  
🔥 단일 API 호출로 다음 작업을 처리할 수 있습니다:

✅ 멀티모달 입력을 사용하여 주어진 이미지를 분석합니다.

✅ `web_search` 호스팅 도구를 통해 웹 검색을 수행합니다.

✅ 결과를 요약합니다.

반면, Chat Completions API를 사용하면 여러 단계가 필요하며, 각 단계는 API로의 왕복 호출을 요구합니다:

1️⃣ 이미지를 업로드하고 분석 결과를 얻음 → 1회 요청

2️⃣ 정보를 추출하고 외부 웹 검색 호출 → 수동 단계 + 도구 실행

3️⃣ 도구 결과를 요약을 위해 다시 제출 → 추가 요청

다음 다이어그램에서 시각적으로 비교한 내용을 확인하세요!

![Responses vs Completions](../../images/comparisons.png)

Responses API를 사용해 보고 코드 단순화와 복잡한 멀티모달, 도구 확장 상호작용을 구축하는 데 얼마나 유용한지 확인해 보세요!

## Responses API란 무엇인가요?

Responses API는 OpenAI 모델과 상호작용하는 새로운 방식으로, 이전 API보다 더 간단하고 유연하게 설계되었습니다. 이 API는 여러 도구를 사용하고, 다중 턴 대화를 처리하며, 텍스트뿐만 아니라 다양한 유형의 데이터를 다룰 수 있는 고급 AI 애플리케이션을 쉽게 구축할 수 있도록 합니다.

Chat Completions와 같이 주로 텍스트를 위해 설계된 이전 API나 설정이 많이 필요한 Assistants API와 달리, Responses API는 다음을 위해 처음부터 설계되었습니다:

- 원활한 다중 턴 상호작용 (단일 API 호출에서 여러 단계의 대화를 이어갈 수 있음)
- 강력한 호스팅 도구에 대한 쉬운 접근 (파일 검색, 웹 검색, 코드 해석기 등)
- 모델에 보낼 컨텍스트에 대한 세밀한 제어

AI 모델이 점점 더 복잡하고 장기적인 추론을 수행할 수 있게 되면서, 비동기적이고 상태를 유지할 수 있는 API가 필요합니다. Responses API는 이러한 요구를 충족하도록 설계되었습니다.

이 가이드에서는 Responses API가 제공하는 새로운 기능과 함께 시작하는 데 도움이 되는 실용적인 예제를 살펴봅니다.

## 기본 사항

Responses API는 표면적으로 Completions API와 매우 유사하게 설계되었습니다.

In [ ]:
from openai import AzureOpenAI
import os
from dotenv import load_dotenv
load_dotenv(override=True)

client = AzureOpenAI(
  azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"), 
  api_key=os.getenv("AZURE_OPENAI_KEY"),  
  api_version="2025-03-01-preview"
)

In [ ]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="농담 하나 해줘",
)

In [ ]:
print(response.output[0].content[0].text)

Responses API의 주요 기능 중 하나는 상태를 유지한다는 점입니다. 즉, 대화의 상태를 직접 관리할 필요가 없으며, API가 이를 대신 처리합니다. 예를 들어, 언제든지 응답을 가져오면 전체 대화 기록이 포함됩니다.

In [ ]:
fetched_response = client.responses.retrieve(
    response_id=response.id
)

print(fetched_response.output[0].content[0].text)

이전 응답을 참조하여 대화를 계속할 수 있습니다.

In [ ]:
response_two = client.responses.create(
    model="gpt-4o-mini",
    input="다른 농담 하나 더 해줘",
    previous_response_id=response.id
)

In [ ]:
print(response_two.output[0].content[0].text)

물론 컨텍스트를 직접 관리할 수도 있습니다. 하지만 OpenAI가 컨텍스트를 유지해 주는 이점 중 하나는 응답을 원하는 시점에서 분기(fork)하여 해당 지점에서 대화를 계속할 수 있다는 점입니다.

In [ ]:
response_two_forked = client.responses.create(
    model="gpt-4o-mini",
    input="그 농담은 별로였어. 다른 농담 하나 더 해주고 두 농담의 차이점도 알려줘",
    previous_response_id=response.id  # 첫 번째 응답에서 분기하여 계속 진행
)

output_text = response_two_forked.output[0].content[0].text
print(output_text)

## 호스팅 도구

Responses API의 또 다른 이점은 `file_search` 및 `web_search`와 같은 호스팅 도구를 지원한다는 점입니다. 도구를 수동으로 호출하는 대신, 도구를 전달하면 API가 어떤 도구를 사용할지 결정하고 이를 사용합니다.

다음은 `web_search` 도구를 사용하여 웹 검색 결과를 응답에 통합하는 예제입니다.

In [ ]:
client = AzureOpenAI(
  azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"), 
  api_key=os.getenv("AZURE_OPENAI_KEY"),  
  api_version="2025-03-01-preview"
)

response = client.responses.create(
    model="gpt-4o",  # 또는 다른 지원 모델
    input="AI에 대한 최신 뉴스는 무엇인가요?",
    tools=[
        {
            "type": "web_search"
        }
    ]
)

In [ ]:
import json
print(json.dumps(response.output, default=lambda o: o.__dict__, indent=2))

## 멀티모달, 도구 확장 대화

Responses API는 텍스트, 이미지, 오디오 모달리티를 기본적으로 지원합니다.  
모든 것을 결합하여 Responses API를 통해 단일 API 호출로 완전한 멀티모달, 도구 확장 상호작용을 구축할 수 있습니다.

In [ ]:
import base64

from IPython.display import Image, display

# 제공된 URL에서 이미지를 표시
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Cat_August_2010-4.jpg/2880px-Cat_August_2010-4.jpg"
display(Image(url=url, width=400))

response_multimodal = client.responses.create(
    model="gpt-4o",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": 
                 "이미지와 관련된 키워드를 생성하고, 검색 도구를 사용하여 키워드와 관련된 뉴스를 웹에서 검색하세요. "
                 "결과를 요약하고 출처를 인용하세요."},
                {"type": "input_image", "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Cat_August_2010-4.jpg/2880px-Cat_August_2010-4.jpg"}
            ]
        }
    ],
    tools=[
        {"type": "web_search"}
    ]
)

In [ ]:
import json

def print_response_text(resp):
    # 1) 가장 간단한 경로: SDK가 합쳐준 텍스트 제공
    if getattr(resp, "output_text", None):
        print(resp.output_text)
        return

    # 2) output 배열에서 message/output_text를 수집
    extracted = []
    for item in getattr(resp, "output", []) or []:
        if getattr(item, "type", None) == "message":
            for content in getattr(item, "content", []) or []:
                if getattr(content, "type", None) == "output_text":
                    text = getattr(content, "text", "")
                    if text:
                        extracted.append(text)

    if extracted:
        print("\n\n".join(extracted))
    else:
        # 3) 마지막 fallback: 한글이 보이도록 ensure_ascii=False
        print(json.dumps(resp.__dict__, default=lambda o: o.__dict__, indent=2, ensure_ascii=False))

print_response_text(response_multimodal)

위 예제에서는 `web_search` 도구를 사용하여 단일 API 호출로 이미지와 관련된 뉴스를 검색할 수 있었습니다. 이는 Chat Completions API를 사용할 경우 여러 번의 왕복 호출이 필요했던 작업을 단순화합니다.

Responses API를 사용하면  
🔥 단일 API 호출로 다음 작업을 처리할 수 있습니다:

✅ 멀티모달 입력을 사용하여 주어진 이미지를 분석합니다.

✅ `web_search` 호스팅 도구를 통해 웹 검색을 수행합니다.

✅ 결과를 요약합니다.

반면, Chat Completions API를 사용하면 여러 단계가 필요하며, 각 단계는 API로의 왕복 호출을 요구합니다:

1️⃣ 이미지를 업로드하고 분석 결과를 얻음 → 1회 요청

2️⃣ 정보를 추출하고 외부 웹 검색 호출 → 수동 단계 + 도구 실행

3️⃣ 도구 결과를 요약을 위해 다시 제출 → 추가 요청

다음 다이어그램에서 시각적으로 비교한 내용을 확인하세요!

![Responses vs Completions](../../images/comparisons.png)

Responses API를 사용해 보고 코드 단순화와 복잡한 멀티모달, 도구 확장 상호작용을 구축하는 데 얼마나 유용한지 확인해 보세요!